# Loading the Data in

In [15]:
# Apply displacement vectors from small MC ROI to large ROI using B-mode tracking
import numpy as np
import nibabel as nib
from scipy.ndimage import shift as scipy_shift, gaussian_filter
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load your B-mode data (for motion tracking)
# Use the normalized version instead of RAW if there are loading issues
bmode_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_Fund_RAW.nii'
ceus_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_CHI_RAW.nii'
print(f"Loading B-mode from: {bmode_path}")

try:
    bmode_data = nib.load(bmode_path).get_fdata()
except Exception as e:
    print(f"Error loading with nibabel: {e}")
    print("Trying manual load...")
    # Manual NIfTI load as fallback
    import struct
    with open(bmode_path, 'rb') as f:
        header = f.read(352)
        dim = struct.unpack('<8h', header[40:56])
        datatype = struct.unpack('<h', header[70:72])[0]
        
        # Read the rest of the file
        data_bytes = f.read()
        
        # Parse based on datatype (64 = float64, 16 = float32, 4 = int16, 2 = uint8)
        if datatype == 64:
            dtype = np.float64
        elif datatype == 16:
            dtype = np.float32
        elif datatype == 4:
            dtype = np.int16
        elif datatype == 2:
            dtype = np.uint8
        else:
            raise ValueError(f"Unknown datatype: {datatype}")
        
        # Reshape
        shape = [d for d in dim[1:dim[0]+1] if d > 1]
        bmode_data = np.frombuffer(data_bytes, dtype=dtype).reshape(shape)

print(f"B-mode data shape: {bmode_data.shape}")
print(f"B-mode data dtype: {bmode_data.dtype}")

# ============================================================================
# LOAD CEUS DATA FOR VISUALIZATION
# ============================================================================
# UPDATE THIS PATH to your CEUS data, or leave as empty string '' to use B-mode
if ceus_path:
    print(f"\nLoading CEUS data from: {ceus_path}")
    try:
        ceus_data = nib.load(ceus_path).get_fdata()
        print(f"CEUS data shape: {ceus_data.shape}")
        viz_data = ceus_data
        viz_data_name = "CEUS"
    except Exception as e:
        print(f"Error loading CEUS: {e}")
        print("Falling back to B-mode for visualization")
        viz_data = bmode_data
        viz_data_name = "B-mode"
else:
    print("\nNo CEUS path provided, using B-mode for visualization")
    viz_data = bmode_data
    viz_data_name = "B-mode"

# Load your small motion-corrected ROI (defines the tracking region)
small_mc_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_small_mc_roi.nii.gz'  # UPDATE THIS PATH
small_mc_roi_seg = nib.load(small_mc_roi_path).get_fdata()

# Load your large (full lesion) ROI  
large_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_static_roi_139.nii.gz'  # UPDATE THIS PATH
large_roi_seg = nib.load(large_roi_path).get_fdata()

print(f"\nSmall MC ROI shape (raw): {small_mc_roi_seg.shape}")
print(f"Large ROI shape (raw): {large_roi_seg.shape}")


Loading B-mode from: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_Fund_RAW.nii
B-mode data shape: (604, 524, 873)
B-mode data dtype: float64

Loading CEUS data from: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_CHI_RAW.nii
CEUS data shape: (604, 524, 873)

Small MC ROI shape (raw): (1048, 604, 873)
Large ROI shape (raw): (1048, 604, 873)


# Separate the ROI for CEUS image analysis

In [16]:
# Process ROIs if they're from combined side-by-side visualization
def process_combined_roi(roi_mask, target_shape):
    """Split side-by-side mask and transpose to match target shape."""
    if roi_mask.shape[0] == 1048:  # Combined side-by-side
        print(f"  Detected combined side-by-side ROI, splitting...")
        mid = 524
        left_half = roi_mask[:mid, :, :]
        right_half = roi_mask[mid:, :, :]
        
        # Choose half with more ROI data
        if np.sum(left_half > 0) > np.sum(right_half > 0):
            roi_mask = left_half
        else:
            roi_mask = right_half
        
        # Transpose to match image orientation
        if roi_mask.shape != target_shape:
            roi_mask = roi_mask.transpose(1, 0, 2)
    
    return roi_mask

print(f"Original B-mode shape: {bmode_data.shape}")

# Process visualization data if needed
if viz_data.shape[0] == 1048:
    print(f"Processing combined CEUS data...")
    viz_data = process_combined_roi(viz_data, bmode_data.shape)
    print(f"Processed CEUS shape: {viz_data.shape}")

# Process ROIs to match original bmode shape
target_shape = (bmode_data.shape[0], bmode_data.shape[1], bmode_data.shape[2])
small_mc_roi_seg = process_combined_roi(small_mc_roi_seg, target_shape)
large_roi_seg = process_combined_roi(large_roi_seg, target_shape)

print(f"\nFinal shapes (all in Y, X, T format):")
print(f"  B-mode data: {bmode_data.shape}")
print(f"  Visualization data ({viz_data_name}): {viz_data.shape}")
print(f"  Small MC ROI: {small_mc_roi_seg.shape}")
print(f"  Large ROI: {large_roi_seg.shape}")

n_frames = bmode_data.shape[2]
print(f"  Number of frames: {n_frames}")

Original B-mode shape: (604, 524, 873)
  Detected combined side-by-side ROI, splitting...
  Detected combined side-by-side ROI, splitting...

Final shapes (all in Y, X, T format):
  B-mode data: (604, 524, 873)
  Visualization data (CEUS): (604, 524, 873)
  Small MC ROI: (604, 524, 873)
  Large ROI: (604, 524, 873)
  Number of frames: 873


# Track the motion compensation bounding box and apply to the actual ROI

In [17]:
def visualize_comparison(frame_idx,fig_compare,axes_compare,top_left_positions,ref_top_left,large_roi_mc,translation_vectors):
    for ax in axes_compare:
        ax.clear()
    
    axes_compare[0].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[0].contour(small_mc_roi_seg[:, :, frame_idx], colors='green', linewidths=2)
    axes_compare[0].plot(top_left_positions[frame_idx, 1], top_left_positions[frame_idx, 0], 'r*', markersize=20)
    axes_compare[0].plot(ref_top_left[1], ref_top_left[0], 'y*', markersize=15, alpha=0.5)
    axes_compare[0].set_title(f'Small MC ROI (tracking)')
    axes_compare[0].axis('off')
    
    axes_compare[1].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[1].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=2)
    axes_compare[1].set_title(f'Large ROI (static)')
    axes_compare[1].axis('off')
    
    axes_compare[2].imshow(viz_data[:, :, frame_idx], cmap='gray')
    axes_compare[2].contour(large_roi_mc[:, :, frame_idx], colors='blue', linewidths=2)
    axes_compare[2].set_title(f'Large ROI (motion corrected)\ndx={translation_vectors[frame_idx, 0]:.1f}, dy={translation_vectors[frame_idx, 1]:.1f}')
    axes_compare[2].axis('off')
    
    fig_compare.suptitle(f'Frame {frame_idx}')
    fig_compare.tight_layout()


1. Track motion first

In [18]:
# ============================================================================
# TRACK MOTION FROM SMALL MC ROI
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.animation import FFMpegWriter
from scipy.ndimage import binary_fill_holes, binary_dilation, binary_erosion
import os

print("="*70)
print("TRACKING MOTION FROM SMALL MC ROI")
print("="*70)

# Reference frame
reference_frame = 139
n_frames = bmode_data.shape[2]

# Arrays to store results
top_left_positions = np.zeros((n_frames, 2))
translation_vectors = np.zeros((n_frames, 3))  # (dx, dy, time)

# Track top-left corner
print("\nTracking top-left corner...")
frames_with_roi = []
for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Frame {t}/{n_frames}...")
    
    mask_t = small_mc_roi_seg[:, :, t] > 0
    
    if np.sum(mask_t) == 0:
        top_left_positions[t] = [-1, -1]  # Flag as invalid
        continue
    
    frames_with_roi.append(t)
    y_indices, x_indices = np.where(mask_t)
    top_left_positions[t] = [y_indices.min(), x_indices.min()]

# Interpolate missing frames
print(f"\nFound ROI in {len(frames_with_roi)}/{n_frames} frames")
print(f"Empty frames: {n_frames - len(frames_with_roi)}")

if len(frames_with_roi) > 0:
    for t in range(n_frames):
        if top_left_positions[t, 0] == -1:
            valid_before = [f for f in frames_with_roi if f < t]
            valid_after = [f for f in frames_with_roi if f > t]
            
            if valid_before and valid_after:
                t_before = max(valid_before)
                t_after = min(valid_after)
                weight = (t - t_before) / (t_after - t_before)
                top_left_positions[t] = (1 - weight) * top_left_positions[t_before] + weight * top_left_positions[t_after]
            elif valid_before:
                top_left_positions[t] = top_left_positions[max(valid_before)]
            elif valid_after:
                top_left_positions[t] = top_left_positions[min(valid_after)]
            else:
                top_left_positions[t] = [0, 0]

# Calculate translation vectors
ref_top_left = top_left_positions[reference_frame]
print(f"\nReference top-left at: (y={ref_top_left[0]:.1f}, x={ref_top_left[1]:.1f})")

for t in range(n_frames):
    dy = top_left_positions[t, 0] - ref_top_left[0]
    dx = top_left_positions[t, 1] - ref_top_left[1]
    translation_vectors[t] = [dx, dy, t]

print(f"\nTranslation range:")
print(f"  X: [{translation_vectors[:, 0].min():.1f}, {translation_vectors[:, 0].max():.1f}] px")
print(f"  Y: [{translation_vectors[:, 1].min():.1f}, {translation_vectors[:, 1].max():.1f}] px")

# ============================================================================
# FILL LARGE ROI (if needed)
# ============================================================================
print("\n" + "="*70)
print("CHECKING IF LARGE ROI NEEDS FILLING")
print("="*70)

test_frame = min(250, n_frames - 1)
roi_test = large_roi_seg[:, :, test_frame]
if np.sum(roi_test > 0) > 0:
    y_coords, x_coords = np.where(roi_test > 0)
    y_min, y_max = y_coords.min(), y_coords.max()
    x_min, x_max = x_coords.min(), x_coords.max()
    bbox_area = (y_max - y_min + 1) * (x_max - x_min + 1)
    roi_pixels = np.sum(roi_test > 0)
    fill_percentage = (roi_pixels / bbox_area) * 100
    
    print(f"Sample frame {test_frame}: {fill_percentage:.1f}% filled")
    
    if fill_percentage < 30:
        print(f"⚠️  ROI is a BORDER/CONTOUR → Filling to create solid mask")
        needs_filling = True
    else:
        print(f"✓ ROI is already filled → Using as-is")
        needs_filling = False
else:
    needs_filling = False

if needs_filling:
    large_roi_seg_filled = np.zeros_like(large_roi_seg)
    for t in range(n_frames):
        if t % 100 == 0:
            print(f"  Filling frame {t}/{n_frames}...")
        roi_frame = large_roi_seg[:, :, t]
        if np.sum(roi_frame > 0) > 0:
            dilated = binary_dilation(roi_frame > 0, iterations=2)
            filled = binary_fill_holes(dilated)
            filled = binary_erosion(filled, iterations=2)
            large_roi_seg_filled[:, :, t] = filled.astype(np.uint8)
    large_roi_seg = large_roi_seg_filled
    print(f"✓ Filled! Added {np.sum(large_roi_seg > 0) - roi_pixels * n_frames} pixels")
    

# ============================================================================
# APPLY MOTION COMPENSATION TO LARGE ROI
# ============================================================================
print("\n" + "="*70)
print("APPLYING MOTION COMPENSATION TO LARGE ROI")
print("="*70)

large_roi_mc = np.zeros_like(large_roi_seg)

for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Processing frame {t}/{n_frames}...")
    
    shift_y = int(round(translation_vectors[t, 1]))
    shift_x = int(round(translation_vectors[t, 0]))
    
    shifted = large_roi_seg[:, :, t].copy()
    if shift_y != 0:
        shifted = np.roll(shifted, shift_y, axis=0)
        if shift_y > 0:
            shifted[:shift_y, :] = 0
        else:
            shifted[shift_y:, :] = 0
    
    if shift_x != 0:
        shifted = np.roll(shifted, shift_x, axis=1)
        if shift_x > 0:
            shifted[:, :shift_x] = 0
        else:
            shifted[:, shift_x:] = 0
    
    large_roi_mc[:, :, t] = shifted

print(f"\n✓ Motion compensation complete!")
print(f"  Total ROI pixels: {np.sum(large_roi_mc > 0)}")
print(f"  Frames with ROI: {np.sum([np.any(large_roi_mc[:,:,t] > 0) for t in range(n_frames)])}/{n_frames}")


TRACKING MOTION FROM SMALL MC ROI

Tracking top-left corner...
  Frame 0/873...
  Frame 100/873...
  Frame 200/873...
  Frame 300/873...
  Frame 400/873...
  Frame 500/873...
  Frame 600/873...
  Frame 700/873...
  Frame 800/873...

Found ROI in 871/873 frames
Empty frames: 2

Reference top-left at: (y=144.0, x=178.0)

Translation range:
  X: [-119.0, 59.0] px
  Y: [-28.0, 15.0] px

CHECKING IF LARGE ROI NEEDS FILLING
Sample frame 250: 77.8% filled
✓ ROI is already filled → Using as-is

APPLYING MOTION COMPENSATION TO LARGE ROI
  Processing frame 0/873...
  Processing frame 100/873...
  Processing frame 200/873...
  Processing frame 300/873...
  Processing frame 400/873...
  Processing frame 500/873...
  Processing frame 600/873...
  Processing frame 700/873...
  Processing frame 800/873...

✓ Motion compensation complete!
  Total ROI pixels: 34415164
  Frames with ROI: 873/873


2. APPLY MOTION COMPENSATION TO BMODE AND CEUS DATA (simple roll)

In [19]:
bmode_data_mc = np.zeros_like(bmode_data)
ceus_data_mc = np.zeros_like(ceus_data)

for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Processing frame {t}/{n_frames}...")

    shift_y = -int(round(translation_vectors[t, 1]))
    shift_x = -int(round(translation_vectors[t, 0]))

    # B-mode MC
    shifted_bmode = bmode_data[:, :, t].copy()
    if shift_y != 0:
        shifted_bmode = np.roll(shifted_bmode, shift_y, axis=0)
    if shift_x != 0:
        shifted_bmode = np.roll(shifted_bmode, shift_x, axis=1)
    bmode_data_mc[:, :, t] = shifted_bmode

    # CEUS MC
    shifted_ceus = ceus_data[:, :, t].copy()
    if shift_y != 0:
        shifted_ceus = np.roll(shifted_ceus, shift_y, axis=0)
    if shift_x != 0:
        shifted_ceus = np.roll(shifted_ceus, shift_x, axis=1)
    ceus_data_mc[:, :, t] = shifted_ceus

print(f"\n✓ Motion compensation complete!")
print(f"  B-mode MC shape: {bmode_data_mc.shape}")
print(f"  CEUS MC shape: {ceus_data_mc.shape}")

original_ceus = nib.load(ceus_path)
ceus_mc_nii = nib.Nifti1Image(ceus_data_mc, affine=original_ceus.affine, header=original_ceus.header)
output_path = os.path.join(os.path.dirname(small_mc_roi_path), "image_mc_data.nii.gz")
nib.save(ceus_mc_nii, output_path)
print(f"Saved to: {output_path}")

  Processing frame 0/873...
  Processing frame 100/873...
  Processing frame 200/873...
  Processing frame 300/873...
  Processing frame 400/873...
  Processing frame 500/873...
  Processing frame 600/873...
  Processing frame 700/873...
  Processing frame 800/873...

✓ Motion compensation complete!
  B-mode MC shape: (604, 524, 873)
  CEUS MC shape: (604, 524, 873)
Saved to: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/image_mc_data.nii.gz


3. Visualization and Video Export

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

vmin_bmode, vmax_bmode = np.percentile(bmode_data_mc, [1, 99])
vmin_ceus, vmax_ceus = np.percentile(ceus_data, [1, 99])

fig_export, axes_export = plt.subplots(1, 3, figsize=(18, 6))
plt.close(fig_export)

def draw_export_frame(frame_idx):
    for ax in axes_export:
        ax.clear()
    axes_export[0].imshow(ceus_data[:, :, frame_idx], cmap='gray', vmin=vmin_ceus, vmax=vmax_ceus)
    axes_export[0].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=1.5)
    axes_export[0].set_title('Original CEUS', fontsize=14)
    axes_export[0].axis('off')
    axes_export[1].imshow(bmode_data_mc[:, :, frame_idx], cmap='gray', vmin=vmin_bmode, vmax=vmax_bmode)
    axes_export[1].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=1.5)
    axes_export[1].set_title('B-mode (MC)', fontsize=14)
    axes_export[1].axis('off')
    axes_export[2].imshow(ceus_data_mc[:, :, frame_idx], cmap='gray', vmin=vmin_ceus, vmax=vmax_ceus)
    axes_export[2].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=1.5)
    axes_export[2].set_title('CEUS (MC)', fontsize=14)
    axes_export[2].axis('off')
    fig_export.suptitle(f'Frame {frame_idx}', fontsize=16, fontweight='bold')
    fig_export.tight_layout()

def do_export(btn):
    output_dir = os.path.dirname(small_mc_roi_path)
    output_path = os.path.join(output_dir, "image_correction.mp4")
    with export_output:
        clear_output()
        print(f"Exporting to: {output_path}")
        writer = FFMpegWriter(fps=10)
        with writer.saving(fig_export, output_path, dpi=100):
            for t in range(n_frames):
                if t % 100 == 0:
                    print(f"  Frame {t}/{n_frames}...")
                draw_export_frame(t)
                writer.grab_frame()
        print(f"\nDone! Saved to: {output_path}")

frame_slider = widgets.IntSlider(value=reference_frame, min=0, max=n_frames-1, step=1, description='Frame:', continuous_update=False)
export_btn = widgets.Button(description='Export Video', button_style='success')
preview_output = widgets.Output()
export_output = widgets.Output()

def on_frame_change(change):
    with preview_output:
        clear_output(wait=True)
        draw_export_frame(change['new'])
        display(fig_export)

frame_slider.observe(on_frame_change, names='value')
export_btn.on_click(do_export)

display(widgets.HBox([frame_slider, export_btn]), preview_output, export_output)
frame_slider.value = reference_frame

Output()

Output()

3. Saving the results

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================
# roi_dir = os.path.dirname(large_roi_path)
# roi_filename = os.path.basename(large_roi_path)
# roi_name_no_ext = roi_filename.replace('.nii.gz', '').replace('.nii', '')

# output_filename = f"{roi_name_no_ext}_displacement_mc.nii.gz"
# output_path = os.path.join(roi_dir, output_filename)

# import nibabel as nib
# nib.save(nib.Nifti1Image(large_roi_mc.astype(np.uint8), np.eye(4)), output_path)
# print(f"✓ Saved to: {output_path}")

# vectors_filename = f"{roi_name_no_ext}_translation_vectors.npy"
# vectors_path = os.path.join(roi_dir, vectors_filename)
# np.save(vectors_path, translation_vectors)
# print(f"✓ Translation vectors saved to: {vectors_path}")